In [9]:
import pandas as pd
import numpy as np
import io
import os

# ============================================================================
# 0. SET YOUR PATHS HERE
# ============================================================================
# Folder containing all your raw CSVs (adjust if your files live elsewhere)
DATA_DIR = r"C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\dataset"

path = DATA_DIR + os.sep

COMMON_COLS = [
    "source_dataset", "date", "time", "vehicle_type",
    "pickup_location", "drop_location",
    "distance_km", "fare_amount", "payment_method",
    "status", "driver_rating", "customer_rating"
]

def make_common(df):
    for c in COMMON_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[COMMON_COLS]

# ============================================================================
# 1. LOAD RAW FILES
# ============================================================================
print("Loading raw files from:", DATA_DIR)

df_ola      = pd.read_csv(path + "Bengaluru_ola.csv")
df_bookings = pd.read_csv(path + "Bookings.csv")
df_ncr      = pd.read_csv(path + "ncr_rides_final_event_dataset_2024.csv")
df_rides    = pd.read_csv(path + "rides_data.csv")
df_delhi    = pd.read_csv(path + "delhi_fare_rates.csv")
df_aru      = pd.read_csv(path + "arunachal_fare_rates.csv")

# --- Indore Ola needs a special loader (rows are double-quote wrapped) ---
def load_indore_ola(filepath):
    with open(filepath, "r", encoding="utf-8", newline="") as f:
        raw_lines = f.read().splitlines()
    cleaned_lines = []
    for ln in raw_lines:
        ln = ln.strip()
        if ln.startswith('"') and ln.endswith('"'):
            ln = ln[1:-1]
        cleaned_lines.append(ln)
    return pd.read_csv(io.StringIO("\n".join(cleaned_lines)))

df_indore = load_indore_ola(path + "indore_ola_dataset.csv")

# ============================================================================
# 2. CLEAN + STANDARDIZE EACH DATASET INDIVIDUALLY
# ============================================================================

# --- 2a. Bengaluru_ola.csv ---
d = df_ola.copy()
d["source_dataset"]   = "bengaluru_ola"
d["date"]             = pd.to_datetime(d["Date"], format="%d/%m/%Y", errors="coerce")
d["time"]             = d["Time"]
d["vehicle_type"]     = d["Vehicle Type"]
d["pickup_location"]  = d["Pickup Location"]
d["drop_location"]    = d["Drop Location"]
d["distance_km"]      = pd.to_numeric(d["Ride Distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["Booking Value"], errors="coerce")
d["payment_method"]   = d["Payment Method"]
d["status"]           = d["Booking Status"]
d["driver_rating"]    = pd.to_numeric(d["Driver Ratings"], errors="coerce")
d["customer_rating"]  = pd.to_numeric(d["Customer Rating"], errors="coerce")
df_ola_clean = make_common(d)

# --- 2b. Bookings.csv ---
d = df_bookings.copy()
d.columns = d.columns.str.strip()
d["source_dataset"]   = "bookings"
d["date"]             = pd.to_datetime(d["Date"], errors="coerce")
d["time"]             = d["Time"]
d["vehicle_type"]     = d["Vehicle_Type"]
d["pickup_location"]  = d["Pickup_Location"]
d["drop_location"]    = d["Drop_Location"]
d["distance_km"]      = pd.to_numeric(d["Ride_Distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["Booking_Value"], errors="coerce")
d["payment_method"]   = d["Payment_Method"]
d["status"]           = d["Booking_Status"]
d["driver_rating"]    = pd.to_numeric(d["Driver_Ratings"], errors="coerce")
d["customer_rating"]  = pd.to_numeric(d["Customer_Rating"], errors="coerce")
df_bookings_clean = make_common(d)

# --- 2c. ncr_rides_final_event_dataset_2024.csv ---
d = df_ncr.copy()
d["source_dataset"]   = "ncr_events"
d["date"]             = pd.to_datetime(d["Date"], errors="coerce")
d["time"]             = d["Time"]
d["vehicle_type"]     = d["Vehicle Type"]
d["pickup_location"]  = d["Pickup Location"]
d["drop_location"]    = d["Drop Location"]
d["distance_km"]      = pd.to_numeric(d["Ride Distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["Booking Value"], errors="coerce")
d["payment_method"]   = d["Payment Method"]
d["status"]           = d["Booking Status"]
d["driver_rating"]    = pd.to_numeric(d["Driver Ratings"], errors="coerce")
d["customer_rating"]  = pd.to_numeric(d["Customer Rating"], errors="coerce")
df_ncr_clean = make_common(d)

# --- 2d. rides_data.csv ---
d = df_rides.copy()
d["source_dataset"]   = "rides_data"
d["date"]             = pd.to_datetime(d["date"], errors="coerce")
d["time"]             = d["time"]
d["vehicle_type"]     = d["services"]
d["pickup_location"]  = d["source"]
d["drop_location"]    = d["destination"]
d["distance_km"]      = pd.to_numeric(d["distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["total_fare"], errors="coerce")
d["payment_method"]   = d["payment_method"]
d["status"]           = d["ride_status"]
df_rides_clean = make_common(d)

# --- 2e. Delhi official fare-rate table ---
d = df_delhi.copy()
d["source_dataset"]  = "delhi_notification_2023"
d["date"]            = pd.NaT
d["time"]            = np.nan
d["vehicle_type"] = np.where(
    d["vehicle_type"] == "taxi",
    "taxi_" + d["ac_type"].astype(str),
    d["vehicle_type"]
)
d["pickup_location"] = "NCT of Delhi (official rate card)"
d["drop_location"]   = np.nan
d["distance_km"]     = pd.to_numeric(d["distance_km"], errors="coerce")
d["fare_amount"]     = pd.to_numeric(d["fare_amount"], errors="coerce")
d["payment_method"]  = np.nan
d["status"]          = "official_rate_card"
d["driver_rating"]   = np.nan
d["customer_rating"] = np.nan
df_delhi_clean = make_common(d)

# --- 2f. Arunachal Pradesh route-fare table ---
d = df_aru.copy()
d["source_dataset"]  = "aru_dto_2019"
d["date"]            = pd.NaT
d["time"]            = np.nan
d["vehicle_type"]    = d["vehicle_type"]
d["pickup_location"] = d["from_stand"]
d["drop_location"]   = d["to_location"]
d["distance_km"]     = np.nan
d["fare_amount"]     = pd.to_numeric(d["fare_amount"], errors="coerce")
d["payment_method"]  = np.nan
d["status"]          = d["fare_type"]
d["driver_rating"]   = np.nan
d["customer_rating"] = np.nan
df_aru_clean = make_common(d)

# --- 2g. Indore Ola dataset ---
d = df_indore.copy()
d.columns = d.columns.str.strip()
d["source_dataset"]  = "indore_ola"
d["date"]            = pd.to_datetime(d["Date"], errors="coerce")
d["time"]            = d["Time"]
d["vehicle_type"]    = d["Vehicle Type"]
d["pickup_location"] = d["Pickup Location"]
d["drop_location"]   = d["Drop Location"]
d["distance_km"]     = pd.to_numeric(d["Ride Distance"], errors="coerce")
d["fare_amount"]     = pd.to_numeric(d["Booking Value"], errors="coerce")
d["payment_method"]  = np.nan
d["status"]          = d["Booking Status"]
d["driver_rating"]   = pd.to_numeric(d["Driver Ratings"], errors="coerce")
d["customer_rating"] = pd.to_numeric(d["Customer Rating"], errors="coerce")
df_indore_clean = make_common(d)

# ============================================================================
# 3. MERGE (CONCATENATE) ALL DATASETS INTO ONE
# ============================================================================
merged_df = pd.concat(
    [
        df_ola_clean,
        df_bookings_clean,
        df_ncr_clean,
        df_rides_clean,
        df_delhi_clean,
        df_aru_clean,
        df_indore_clean,
    ],
    ignore_index=True
)

print("Merged shape (before preprocessing):", merged_df.shape)
print(merged_df["source_dataset"].value_counts())

# ============================================================================
# 4. PRE-TRAINING PREPROCESSING
# ============================================================================

# 4a. Drop rows with no fare (target variable)
# merged_df = merged_df.dropna(subset=["fare_amount"])

# 4b. Remove impossible / junk values
merged_df = merged_df[merged_df["fare_amount"] > 0]
merged_df = merged_df[(merged_df["distance_km"].isna()) | (merged_df["distance_km"] >= 0)]

# 4c. Standardize text columns
for col in ["vehicle_type", "payment_method", "status"]:
    merged_df[col] = merged_df[col].astype(str).str.strip().str.lower()
    merged_df[col] = merged_df[col].replace({"nan": np.nan})

# 4d. Feature engineering from date/time
merged_df["date"] = pd.to_datetime(merged_df["date"], errors="coerce")
merged_df["year"]    = merged_df["date"].dt.year
merged_df["month"]   = merged_df["date"].dt.month
merged_df["day"]     = merged_df["date"].dt.day
merged_df["weekday"] = merged_df["date"].dt.day_name()

merged_df["time"] = pd.to_datetime(merged_df["time"], format="%H:%M:%S", errors="coerce").dt.hour
merged_df.rename(columns={"time": "hour"}, inplace=True)

# 4e. Handle duplicates
# merged_df = merged_df.drop_duplicates()

# 4f. Handle remaining missing values
merged_df["distance_km"]     = merged_df["distance_km"].fillna(merged_df["distance_km"].median())
merged_df["driver_rating"]   = merged_df["driver_rating"].fillna(merged_df["driver_rating"].median())
merged_df["customer_rating"] = merged_df["customer_rating"].fillna(merged_df["customer_rating"].median())
merged_df["payment_method"]  = merged_df["payment_method"].fillna("cash")
merged_df["status"]          = merged_df["status"].fillna("sucess")

# 4g. Outlier removal on fare_amount (IQR method)
# Q1 = merged_df["fare_amount"].quantile(0.25)
# Q3 = merged_df["fare_amount"].quantile(0.75)
# IQR = Q3 - Q1
# lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
# merged_df = merged_df[(merged_df["fare_amount"] >= lower) & (merged_df["fare_amount"] <= upper)]

print("Final shape after preprocessing:", merged_df.shape)
print(merged_df["source_dataset"].value_counts())
print(merged_df["vehicle_type"].value_counts())
print(merged_df.head())

# ============================================================================
# 5. SAVE
# ============================================================================
out_path = path + "merged_cab_fare_data_final.csv"
merged_df.to_csv(out_path, index=False)
print("Saved final merged dataset to:", out_path)

Loading raw files from: C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\dataset
Merged shape (before preprocessing): (465073, 12)
source_dataset
ncr_events                 160609
bookings                   103024
indore_ola                 100000
rides_data                  50000
bengaluru_ola               49999
delhi_notification_2023       928
aru_dto_2019                  513
Name: count, dtype: int64
Final shape after preprocessing: (392242, 16)
source_dataset
ncr_events                 109329
bookings                   103024
indore_ola                 100000
rides_data                  44964
bengaluru_ola               33484
delhi_notification_2023       928
aru_dto_2019                  513
Name: count, dtype: int64
vehicle_type
auto             72174
bike             63670
ebike            41595
prime sedan      34088
prime plus       33874
prime suv        33772
mini             33358
go mini          21815
go sedan         19658
premier sedan    13238
c

In [10]:
import numpy as np
import pandas as pd

# ============================================================================
# COLUMN 1: "state"
# ============================================================================
# Mapped from source_dataset -> the Indian state/region the data represents.
# NOTE: "bookings" and "rides_data" don't state their city inside the CSV
# itself — I've assumed Karnataka (Bengaluru) since that's the common origin
# for these Kaggle datasets, but verify against wherever you downloaded them
# and edit the dict below if wrong.

state_map = {
    "bengaluru_ola":            "Karnataka",
    "bookings":                 "Karnataka",       # <-- verify, assumption
    "ncr_events":               "Delhi NCR",       # spans Delhi/Haryana/UP
    "rides_data":               "Karnataka",       # <-- verify, assumption
    "delhi_notification_2023":  "Delhi",
    "aru_dto_2019":             "Arunachal Pradesh",
    "indore_ola":               "Madhya Pradesh",
}

merged_df["state"] = merged_df["source_dataset"].map(state_map)
merged_df["state"] = merged_df["state"].fillna("unknown")


# ============================================================================
# COLUMN 2: "season"  (Indian meteorological seasons, based on month)
# ============================================================================
# Winter      : Dec, Jan, Feb
# Summer      : Mar, Apr, May
# Monsoon     : Jun, Jul, Aug, Sep
# Post-monsoon: Oct, Nov

def month_to_season(m):
    if pd.isna(m):
        return np.nan
    m = int(m)
    if m in [12, 1, 2]:
        return "winter"
    elif m in [3, 4, 5]:
        return "summer"
    elif m in [6, 7, 8, 9]:
        return "monsoon"
    elif m in [10, 11]:
        return "post_monsoon"
    return np.nan

merged_df["season"] = merged_df["month"].apply(month_to_season)
merged_df["season"] = merged_df["season"].fillna("unknown")


# ============================================================================
# COLUMN 3: "weather" (sunny / raining) — ONLY meaningful within monsoon season
# ============================================================================
# Logic: fare typically scales with distance (fare ~ base + rate*distance).
# So fare_per_km = fare_amount / distance_km approximates that per-km rate.
# During monsoon, surge pricing pushes fare_per_km above the "normal" rate.
# We compute the mean fare_per_km across ALL monsoon rows, then:
#   fare_per_km > mean  -> "raining" (surge-like pricing)
#   fare_per_km <= mean -> "sunny"
# Non-monsoon rows get "not_applicable".

merged_df["fare_per_km"] = merged_df["fare_amount"] / merged_df["distance_km"].replace(0, np.nan)

monsoon_mask = merged_df["season"] == "monsoon"
monsoon_mean_fare_per_km = merged_df.loc[monsoon_mask, "fare_per_km"].mean()

print("Monsoon mean fare/km:", monsoon_mean_fare_per_km)

merged_df["weather"] = "not_applicable"
merged_df.loc[monsoon_mask & (merged_df["fare_per_km"] > monsoon_mean_fare_per_km), "weather"] = "raining"
merged_df.loc[monsoon_mask & (merged_df["fare_per_km"] <= monsoon_mean_fare_per_km), "weather"] = "sunny"
merged_df.loc[monsoon_mask & (merged_df["fare_per_km"].isna()), "weather"] = "unknown"

# fare_per_km was only a helper column — drop it unless you want to keep it
merged_df = merged_df.drop(columns=["fare_per_km"])


# ============================================================================
# COLUMN 4: "phase_of_day"  (based on "hour" column)
# ============================================================================
# before_sunrise : 4  - 6
# morning        : 6  - 12
# afternoon      : 12 - 16
# evening        : 16 - 19
# night          : 19 - 24
# midnight       : 0  - 4
#
# THE MISSING-TIME PROBLEM:
# Your PDF-derived rows (delhi_notification_2023, aru_dto_2019) and any row
# with a bad/unparseable time have NO "hour" value — they're rate cards, not
# timestamped trips, so there is no real time to recover for them.
#
# Rather than inventing a time (which would fabricate data that never
# existed), the honest options are:
#   (a) Label these as "unknown" — recommended, keeps the model honest.
#   (b) OPTIONAL: probabilistically impute a phase by sampling from the
#       observed phase_of_day distribution of rows that DO have real
#       timestamps. This fills the column for ML pipelines that can't
#       handle NaN/"unknown" as a category, but it is synthetic data —
#       flagged via a separate is_phase_imputed column so you can exclude
#       it later or treat it differently during training.
#
# Both are implemented below — (a) always runs, (b) is optional (toggle the
# IMPUTE_MISSING_PHASE flag).

def hour_to_phase(h):
    if pd.isna(h):
        return np.nan
    h = int(h)
    if 0 <= h < 4:
        return "midnight"
    elif 4 <= h < 6:
        return "before_sunrise"
    elif 6 <= h < 12:
        return "morning"
    elif 12 <= h < 16:
        return "afternoon"
    elif 16 <= h < 19:
        return "evening"
    elif 19 <= h <= 23:
        return "night"
    return np.nan

merged_df["phase_of_day"] = merged_df["hour"].apply(hour_to_phase)
merged_df["is_phase_imputed"] = merged_df["phase_of_day"].isna()  # True = no real timestamp existed

IMPUTE_MISSING_PHASE = True  # set False if you'd rather just keep "unknown"

if IMPUTE_MISSING_PHASE:
    known_phase_dist = merged_df.loc[~merged_df["is_phase_imputed"], "phase_of_day"].value_counts(normalize=True)
    missing_mask = merged_df["phase_of_day"].isna()
    n_missing = missing_mask.sum()

    if n_missing > 0 and not known_phase_dist.empty:
        rng = np.random.default_rng(42)  # seeded for reproducibility
        sampled_phases = rng.choice(
            known_phase_dist.index,
            size=n_missing,
            p=known_phase_dist.values
        )
        merged_df.loc[missing_mask, "phase_of_day"] = sampled_phases
else:
    merged_df["phase_of_day"] = merged_df["phase_of_day"].fillna("unknown")

print(merged_df[["state", "season", "weather", "phase_of_day", "is_phase_imputed"]].sample(10))
print(merged_df["state"].value_counts())
print(merged_df["season"].value_counts())
print(merged_df["weather"].value_counts())
print(merged_df["phase_of_day"].value_counts())

Monsoon mean fare/km: 48.26453153829001
                 state   season         weather phase_of_day  is_phase_imputed
151207       Karnataka  monsoon           sunny        night             False
292470       Delhi NCR   summer  not_applicable      evening             False
336647       Karnataka  monsoon           sunny      morning              True
6688         Karnataka   winter  not_applicable      morning             False
152640       Karnataka  monsoon         unknown    afternoon             False
378085  Madhya Pradesh   winter  not_applicable      evening              True
113183       Karnataka  monsoon         unknown      evening             False
377169  Madhya Pradesh   winter  not_applicable      morning              True
463028  Madhya Pradesh   winter  not_applicable        night              True
459883  Madhya Pradesh   winter  not_applicable      morning              True
state
Karnataka            181472
Delhi NCR            109329
Madhya Pradesh       100000
D

In [11]:
merged_df

,source_dataset,date,hour,vehicle_type,pickup_location,drop_location,distance_km,fare_amount,payment_method,status,...,customer_rating,year,month,day,weekday,state,season,weather,phase_of_day,is_phase_imputed
0,bengaluru_ola,2024-01-28,6.0,auto,Area-3,Area-2,28.50,868.06,wallet,success,...,4.4,2024.0,1.0,28.0,Sunday,Karnataka,winter,not_applicable,morning,False
6,bengaluru_ola,2024-01-17,21.0,prime sedan,Area-38,Area-26,25.18,348.04,card,success,...,4.7,2024.0,1.0,17.0,Wednesday,Karnataka,winter,not_applicable,night,False
7,bengaluru_ola,2024-01-30,0.0,ebike,Area-47,Area-8,17.67,56.33,upi,success,...,3.3,2024.0,1.0,30.0,Tuesday,Karnataka,winter,not_applicable,midnight,False
8,bengaluru_ola,2024-01-22,8.0,prime suv,Area-11,Area-35,24.94,1971.38,upi,success,...,3.7,2024.0,1.0,22.0,Monday,Karnataka,winter,not_applicable,morning,False
10,bengaluru_ola,2024-01-12,7.0,prime sedan,Area-43,Area-42,13.15,1130.15,card,success,...,3.0,2024.0,1.0,12.0,Friday,Karnataka,winter,not_applicable,morning,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465068,indore_ola,2025-01-02,NaN,prime sedan,Scheme No. 54,MR 10,17.73,908.00,cash,canceled by driver,...,4.2,2025.0,1.0,2.0,Thursday,Madhya Pradesh,winter,not_applicable,evening,True
465069,indore_ola,2025-01-30,NaN,bike,RNT Marg,Patel Nagar,18.34,676.00,cash,success,...,3.7,2025.0,1.0,30.0,Thursday,Madhya Pradesh,winter,not_applicable,evening,True
465070,indore_ola,2025-01-08,NaN,prime plus,Scheme No. 114,South Tukoganj,9.28,780.00,cash,canceled by driver,...,4.2,2025.0,1.0,8.0,Wednesday,Madhya Pradesh,winter,not_applicable,night,True
465071,indore_ola,2025-01-12,NaN,mini,MR 4,Chhoti Gwaltoli,3.32,1852.00,cash,canceled by driver,...,4.2,2025.0,1.0,12.0,Sunday,Madhya Pradesh,winter,not_applicable,night,True


In [6]:
merged_df.to_csv(
    r"C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\dataset\merged_dataset.csv",
    index=False
)

In [4]:
merged_df["state"].value_counts()

state
Karnataka            181472
Delhi NCR            109329
Madhya Pradesh       100000
Delhi                   928
Arunachal Pradesh       513
Name: count, dtype: int64

In [5]:
merged_df.isnull().sum()

source_dataset           0
date                  1441
hour                146405
vehicle_type             0
pickup_location          0
drop_location          928
distance_km              0
fare_amount              0
payment_method           0
status                   0
driver_rating            0
customer_rating          0
year                  1441
month                 1441
day                   1441
weekday               1441
state                    0
season                   0
weather                  0
phase_of_day             0
is_phase_imputed         0
dtype: int64